# Calculate Mean and Standard Deviation of Metrics

This notebook calculates the mean and standard deviation of accuracy, F1 macro, AUROC, and AUPR from two ALERT result CSV files (TRAM and TRAM2) and creates a new CSV with values in [mean] ± [std] format.

In [1]:
import pandas as pd
import numpy as np
import os

## Load the CSV files

In [2]:
# Define file paths
file1 = '../results/ALERT_Results - TRAM.csv'
file2 = '../results/ALERT_Results - TRAM2.csv'
output_file = '../results/ALERT_Results - Mean_Std.csv'

# Read the CSV files
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

print("TRAM.csv:")
print(df1.head())
print(f"\nShape: {df1.shape}")

print("\nTRAM2.csv:")
print(df2.head())
print(f"\nShape: {df2.shape}")

TRAM.csv:
                                Budget      AL Strategy  Accuracy  F1 (macro)  \
0  B1 50 Annotation Cycle, 600 samples   Top Confidence     72.91       60.86   
1  B1 50 Annotation Cycle, 600 samples      Max Entropy     69.50       56.58   
2  B1 50 Annotation Cycle, 600 samples           Energy     73.54       64.15   
3  B1 50 Annotation Cycle, 600 samples  Margin Sampling     76.24       65.24   
4  B1 50 Annotation Cycle, 600 samples       MC Dropout     69.63       58.95   

   AUROC   AUPR  
0  97.90  74.23  
1  97.64  71.45  
2  97.98  73.64  
3  97.91  74.04  
4  96.98  70.07  

Shape: (32, 6)

TRAM2.csv:
                                Budget     AL Strategy   Accuracy  F1 (macro)  \
0  B1 50 Annotation Cycle, 600 samples  Top Confidence  73.090744   67.657197   
1  B1 50 Annotation Cycle, 600 samples         Coreset  66.756517   54.018080   
2  B1 50 Annotation Cycle, 600 samples      MC Dropout  69.631624   58.953977   
3  B1 50 Annotation Cycle, 600 samples     

## Combine the data and calculate mean and std

In [3]:
# Add a source identifier for tracking
df1['Source'] = 'TRAM'
df2['Source'] = 'TRAM2'

# Combine both dataframes
combined_df = pd.concat([df1, df2], ignore_index=True)

print("Combined DataFrame:")
print(combined_df.head(10))
print(f"\nShape: {combined_df.shape}")

Combined DataFrame:
                                  Budget      AL Strategy  Accuracy  \
0    B1 50 Annotation Cycle, 600 samples   Top Confidence     72.91   
1    B1 50 Annotation Cycle, 600 samples      Max Entropy     69.50   
2    B1 50 Annotation Cycle, 600 samples           Energy     73.54   
3    B1 50 Annotation Cycle, 600 samples  Margin Sampling     76.24   
4    B1 50 Annotation Cycle, 600 samples       MC Dropout     69.63   
5    B1 50 Annotation Cycle, 600 samples          Coreset     66.76   
6    B1 50 Annotation Cycle, 600 samples              GMM     74.35   
7    B1 50 Annotation Cycle, 600 samples           Random     73.76   
8  B2 100 Annotation Cycle, 1100 samples   Top Confidence     82.79   
9  B2 100 Annotation Cycle, 1100 samples      Max Entropy     82.61   

   F1 (macro)  AUROC   AUPR Source  
0       60.86  97.90  74.23   TRAM  
1       56.58  97.64  71.45   TRAM  
2       64.15  97.98  73.64   TRAM  
3       65.24  97.91  74.04   TRAM  
4       58.95

In [4]:
# Group by Budget and AL Strategy and calculate mean and std for each metric
grouped = combined_df.groupby(['Budget', 'AL Strategy'])

# Calculate mean and std for each metric
metrics = ['Accuracy', 'F1 (macro)', 'AUROC', 'AUPR']

# Create a new dataframe with mean and std
results = []

for (budget, al_strategy), group in grouped:
    row = {
        'Budget': budget,
        'AL Strategy': al_strategy
    }
    
    for metric in metrics:
        mean_val = group[metric].mean()
        std_val = group[metric].std()
        
        # Format as [mean] ± [std] with 2 decimal places
        row[metric] = f"{mean_val:.2f} ± {std_val:.2f}"
    
    results.append(row)

# Create the final dataframe
result_df = pd.DataFrame(results)

print("\nResults with Mean ± Std:")
print(result_df)


Results with Mean ± Std:
                                   Budget      AL Strategy      Accuracy  \
0     B1 50 Annotation Cycle, 600 samples          Coreset  66.76 ± 0.00   
1     B1 50 Annotation Cycle, 600 samples           Energy  75.56 ± 2.86   
2     B1 50 Annotation Cycle, 600 samples              GMM  74.35 ± 0.00   
3     B1 50 Annotation Cycle, 600 samples       MC Dropout  69.63 ± 0.00   
4     B1 50 Annotation Cycle, 600 samples  Margin Sampling  77.45 ± 1.71   
5     B1 50 Annotation Cycle, 600 samples      Max Entropy  71.90 ± 3.40   
6     B1 50 Annotation Cycle, 600 samples           Random  73.76 ± 0.00   
7     B1 50 Annotation Cycle, 600 samples   Top Confidence  73.00 ± 0.13   
8   B2 100 Annotation Cycle, 1100 samples          Coreset  81.45 ± 0.00   
9   B2 100 Annotation Cycle, 1100 samples           Energy  83.31 ± 0.54   
10  B2 100 Annotation Cycle, 1100 samples              GMM  81.72 ± 0.00   
11  B2 100 Annotation Cycle, 1100 samples       MC Dropout  82

## Display detailed statistics

In [5]:
# Optional: Show detailed statistics for each group
print("\nDetailed Statistics:")
print("=" * 80)

for (budget, al_strategy), group in grouped:
    print(f"\n{budget} - {al_strategy}:")
    for metric in metrics:
        mean_val = group[metric].mean()
        std_val = group[metric].std()
        values = group[metric].values
        print(f"  {metric:15s}: {mean_val:.2f} ± {std_val:.2f}  (values: {values})")


Detailed Statistics:

B1 50 Annotation Cycle, 600 samples - Coreset:
  Accuracy       : 66.76 ± 0.00  (values: [66.76      66.7565167])
  F1 (macro)     : 54.02 ± 0.00  (values: [54.02       54.01808023])
  AUROC          : 96.97 ± 0.00  (values: [96.97       96.97399735])
  AUPR           : 69.13 ± 0.00  (values: [69.13       69.13325191])

B1 50 Annotation Cycle, 600 samples - Energy:
  Accuracy       : 75.56 ± 2.86  (values: [73.54       77.58311033])
  F1 (macro)     : 67.77 ± 5.12  (values: [64.15       71.39037848])
  AUROC          : 98.08 ± 0.14  (values: [97.98       98.17704558])
  AUPR           : 76.33 ± 3.81  (values: [73.64       79.02750969])

B1 50 Annotation Cycle, 600 samples - GMM:
  Accuracy       : 74.35 ± 0.00  (values: [74.35       74.34860468])
  F1 (macro)     : 61.66 ± 0.00  (values: [61.66       61.66083217])
  AUROC          : 96.30 ± 0.00  (values: [96.3        96.30007744])
  AUPR           : 70.37 ± 0.00  (values: [70.37      70.3707695])

B1 50 Annotati

## Save the results to CSV

In [6]:
# Save to CSV
result_df.to_csv(output_file, index=False)

print(f"\nResults saved to: {output_file}")
print(f"\nFinal DataFrame shape: {result_df.shape}")
print("\nPreview of saved file:")
print(result_df.head(10))


Results saved to: ../results/ALERT_Results - Mean_Std.csv

Final DataFrame shape: (32, 6)

Preview of saved file:
                                  Budget      AL Strategy      Accuracy  \
0    B1 50 Annotation Cycle, 600 samples          Coreset  66.76 ± 0.00   
1    B1 50 Annotation Cycle, 600 samples           Energy  75.56 ± 2.86   
2    B1 50 Annotation Cycle, 600 samples              GMM  74.35 ± 0.00   
3    B1 50 Annotation Cycle, 600 samples       MC Dropout  69.63 ± 0.00   
4    B1 50 Annotation Cycle, 600 samples  Margin Sampling  77.45 ± 1.71   
5    B1 50 Annotation Cycle, 600 samples      Max Entropy  71.90 ± 3.40   
6    B1 50 Annotation Cycle, 600 samples           Random  73.76 ± 0.00   
7    B1 50 Annotation Cycle, 600 samples   Top Confidence  73.00 ± 0.13   
8  B2 100 Annotation Cycle, 1100 samples          Coreset  81.45 ± 0.00   
9  B2 100 Annotation Cycle, 1100 samples           Energy  83.31 ± 0.54   

     F1 (macro)         AUROC          AUPR  
0  54.02 ± 0.

## Summary by Budget

In [7]:
# Display results grouped by budget for easier viewing
print("\nResults by Budget:")
print("=" * 100)

for budget in result_df['Budget'].unique():
    print(f"\n{budget}:")
    budget_data = result_df[result_df['Budget'] == budget]
    print(budget_data.to_string(index=False))


Results by Budget:

B1 50 Annotation Cycle, 600 samples:
                             Budget     AL Strategy     Accuracy   F1 (macro)        AUROC         AUPR
B1 50 Annotation Cycle, 600 samples         Coreset 66.76 ± 0.00 54.02 ± 0.00 96.97 ± 0.00 69.13 ± 0.00
B1 50 Annotation Cycle, 600 samples          Energy 75.56 ± 2.86 67.77 ± 5.12 98.08 ± 0.14 76.33 ± 3.81
B1 50 Annotation Cycle, 600 samples             GMM 74.35 ± 0.00 61.66 ± 0.00 96.30 ± 0.00 70.37 ± 0.00
B1 50 Annotation Cycle, 600 samples      MC Dropout 69.63 ± 0.00 58.95 ± 0.00 96.98 ± 0.00 70.07 ± 0.00
B1 50 Annotation Cycle, 600 samples Margin Sampling 77.45 ± 1.71 68.13 ± 4.08 98.00 ± 0.13 76.09 ± 2.90
B1 50 Annotation Cycle, 600 samples     Max Entropy 71.90 ± 3.40 62.90 ± 8.94 97.48 ± 0.22 73.48 ± 2.87
B1 50 Annotation Cycle, 600 samples          Random 73.76 ± 0.00 60.90 ± 0.00 96.78 ± 0.00 71.74 ± 0.00
B1 50 Annotation Cycle, 600 samples  Top Confidence 73.00 ± 0.13 64.26 ± 4.81 97.50 ± 0.56 74.53 ± 0.43

B2 10